# Full architecture demo

This notebook shows the complete workflow we built:
- standardized data loading
- one `Universe` object with shared prices and returns
- multiple constructions stored on the same universe
- per-universe output folders
- automatic export of market data, constructions, backtests, and Monte Carlo results
- fixed-weight backtesting
- Monte Carlo simulation
- Plotly visualization layer
- optional HTML export of figures
- batch export of all available outputs

## Latest Changes Added

The latest additions that were incorporated are:
- `save_construction(...)` and `save_all_constructions()`
- `save_backtest(...)` and `save_all_backtests()`
- `save_monte_carlo(...)` and `save_all_monte_carlo()`
- `PortfolioVisualizer`
- optional `save_html=True` for plots
- batch plot export methods:
  - `save_all_construction_plots()`
  - `save_all_backtest_plots()`
  - `save_all_monte_carlo_plots()`
  - `save_everything()`
- refined Monte Carlo distribution plot
- refined backtest comparison plot

In [2]:
from portafolios import (
    Universe,
    local_loader,
    EqualWeightConstructor,
    Markowitz,
    NaiveRiskParity,
    HRPStyle,
    Backtester,
    MonteCarloEngine,
    PortfolioVisualizer,
)

from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "portafolios").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "processed" / "data_clean_stock_data.csv"
YF_SNAPSHOT_PATH = PROJECT_ROOT / "data" / "yf_snapshot.csv"

print("Project root:", PROJECT_ROOT)
print("Processed CSV exists:", CSV_PATH.exists())
print("Yahoo snapshot exists:", YF_SNAPSHOT_PATH.exists())


Project root: C:\Users\narro\Desktop\semestre\honores_actuaria
Processed CSV exists: True
Yahoo snapshot exists: True


## 1. Create one universe

In [3]:
u = Universe(
    universe_name="thesis_full_demo",
    base_output_dir=PROJECT_ROOT / "outputs" / "runs",
    auto_save_data=True,
    tickers=["AAPL", "MSFT", "AMZN", "GOOG"],
    start="2020-01-01",
    end="2024-12-31",
    construction_start="2020-01-01",
    construction_end="2020-06-30",
    loader=local_loader,
    loader_kwargs={"path": CSV_PATH, "prefer_adj_close": True},
    freq="B",
).preparar_datos()

u.prices.head()


,AAPL,MSFT,AMZN,GOOG
Date,,,,
2020-01-02,72.776606,153.428246,94.900497,68.186813
2020-01-03,72.771729,152.683705,94.309998,68.439404
2020-01-06,72.621654,151.872308,95.184502,69.663459
2020-01-07,72.849231,152.416407,95.694504,69.921535
2020-01-08,73.706271,153.495104,95.550003,70.337515


In [4]:
print("Universe name:", u.universe_name)
print("Output dir:", u.output_dir)
print("Data dir:", u.data_dir)
print("Constructions dir:", u.constructions_dir)
print("Backtests dir:", u.backtests_dir)
print("Monte Carlo dir:", u.monte_carlo_dir)
print("Plots dir:", u.plots_dir)
print("Returns shape:", u.returns.shape)
print("Saved prices exists:", (u.data_dir / "prices.csv").exists())
print("Saved returns exists:", (u.data_dir / "returns.csv").exists())
print("Saved metadata exists:", (u.data_dir / "metadata.json").exists())
print("Saved tickers exists:", (u.data_dir / "tickers.json").exists())

Universe name: thesis_full_demo
Output dir: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\runs\thesis_full_demo
Data dir: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\runs\thesis_full_demo\data
Constructions dir: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\runs\thesis_full_demo\constructions
Backtests dir: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\runs\thesis_full_demo\backtests
Monte Carlo dir: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\runs\thesis_full_demo\monte_carlo
Plots dir: C:\Users\narro\Desktop\semestre\honores_actuaria\outputs\runs\thesis_full_demo\plots
Returns shape: (1303, 4)
Saved prices exists: True
Saved returns exists: True
Saved metadata exists: True
Saved tickers exists: True


## 2. Build multiple constructions on the same universe

In [5]:
construction_kwargs = {
    "ann_factor": 252,
}

u.construir(EqualWeightConstructor(), label="equal_weight", **construction_kwargs)
u.construir(Markowitz(), label="markowitz", ret_kind="simple", allow_short=False, **construction_kwargs)
u.construir(NaiveRiskParity(), label="naive_risk_parity", **construction_kwargs)
u.construir(HRPStyle(), label="hrp_style", **construction_kwargs)

u.list_constructions()


['equal_weight', 'markowitz', 'naive_risk_parity', 'hrp_style']

In [6]:
u.compare_insample_metrics()

,method,display_name,n_selected,expected_return,volatility,sharpe_m,meta_n_assets,meta_success,meta_message,meta_rf_per_period,...,meta_ret_kind_used,meta_nrp_method,meta_nrp_min_vol,meta_nrp_sigma,meta_hrp_n_clusters,meta_hrp_distance,meta_hrp_clustering,meta_hrp_clusters,meta_hrp_inner_n_assets,meta_hrp_outer_n_assets
name,,,,,,,,,,,,,,,,,,,,,
equal_weight,equal_weight,Equal Weight,4,0.486218,0.302358,1.608084,4.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hrp_style,hrp_style,HRP Style,4,0.548192,0.297384,1.843380,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,3.0,corr_distance,hierarchical_clusters,"[[AAPL, GOOG], [MSFT], [AMZN]]",1.0,3.0
markowitz,markowitz_max_sharpe,Markowitz Max Sharpe,2,0.754560,0.316590,2.383402,4.0,True,Optimization terminated successfully,0.0,...,simple,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
naive_risk_parity,naive_risk_parity,Naive Risk Parity (1/sigma),4,0.482342,0.301091,1.601981,NaN,NaN,NaN,NaN,...,NaN,inverse_vol,1.000000e-08,"{'AAPL': 0.024012558289591562, 'MSFT': 0.02147...",NaN,NaN,NaN,NaN,NaN,NaN


## 3. Evaluate the saved constructions

In [18]:
all_bt = Backtester.run_all(
    u,
    start_date="2020-07-01",
    end_date="2020-12-31",
    ann_factor=252,
    attach=True,
)

all_mc = MonteCarloEngine.run_all(
    u,
    horizon=30,
    n_simulations=500,
    initial_value=1.0,
    attach=True,
    seed=123,
)

{name: u.get_construction(name).backtest_result.summary_metrics for name in u.list_constructions() if u.get_construction(name).backtest_result is not None}

{'equal_weight': {'n_periods': 132,
  'total_return': 0.2501002343812391,
  'annualized_return': 0.5313572050716688,
  'annualized_volatility': 0.25583799600161533,
  'sharpe_ratio': 2.0769284210165324,
  'max_drawdown': -0.16550208519812304},
 'markowitz': {'n_periods': 132,
  'total_return': 0.1713666680077497,
  'annualized_return': 0.35251134863049405,
  'annualized_volatility': 0.30595670160893723,
  'sharpe_ratio': 1.152160899816018,
  'max_drawdown': -0.15945368534247528},
 'naive_risk_parity': {'n_periods': 132,
  'total_return': 0.24407861200647418,
  'annualized_return': 0.5173058009586511,
  'annualized_volatility': 0.2543788177470463,
  'sharpe_ratio': 2.0336040773373623,
  'max_drawdown': -0.16454555163092066},
 'hrp_style': {'n_periods': 132,
  'total_return': 0.21439407105039843,
  'annualized_return': 0.4489393879599948,
  'annualized_volatility': 0.25801333126507675,
  'sharpe_ratio': 1.739985239362555,
  'max_drawdown': -0.15961487899086446}}

## 4. Export stored objects to disk

In [19]:
u.save_market_data()
u.save_all_constructions()
u.save_all_backtests()
u.save_all_monte_carlo()

{'equal_weight': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/monte_carlo/equal_weight'),
 'markowitz': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/monte_carlo/markowitz'),
 'naive_risk_parity': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/monte_carlo/naive_risk_parity'),
 'hrp_style': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/monte_carlo/hrp_style')}

In [20]:
print((u.get_construction_dir("markowitz") / "weights.csv").exists())
print((u.get_backtest_dir("markowitz") / "portfolio_returns.csv").exists())
print((u.get_mc_dir("markowitz") / "simulated_paths.csv").exists())

True
True
True


## 5. Build the Plotly visualizer

In [21]:
viz = PortfolioVisualizer(u)
viz

## 6. Construction plots

In [22]:
fig_weights_bar = viz.plot_weights_bar("markowitz")
fig_weights_pie = viz.plot_weights_pie("markowitz")
fig_weights_scatter = viz.plot_weights_scatter("markowitz")

fig_weights_bar

## 7. Refined backtest comparison plot

In [23]:
fig_backtest_comparison = viz.plot_backtest_comparison()
fig_backtest_comparison

## 8. Refined Monte Carlo distribution plot

In [24]:
fig_mc_distribution = viz.plot_mc_distribution("markowitz")
fig_mc_distribution

In [25]:
fig_mc_distribution_oneoversigma = viz.plot_mc_distribution("naive_risk_parity")
fig_mc_distribution_oneoversigma

## 9. Save individual figures as HTML

In [26]:
viz.plot_weights_bar("markowitz", save_html=True)
viz.plot_backtest("markowitz", save_html=True)
viz.plot_mc_distribution("markowitz", save_html=True)
viz.plot_backtest_comparison(save_html=True)

In [15]:
print((u.get_construction_dir("markowitz") / "weights_bar.html").exists())
print((u.get_backtest_dir("markowitz") / "backtest.html").exists())
print((u.get_mc_dir("markowitz") / "mc_distribution.html").exists())
print((u.get_plot_dir() / "backtest_comparison.html").exists())

True
True
True
True


## 10. Batch export all Plotly figures

In [16]:
viz.save_all_construction_plots()
viz.save_all_backtest_plots()
viz.save_all_monte_carlo_plots(max_paths=50)

{'equal_weight': ['mc_paths.html', 'mc_distribution.html'],
 'markowitz': ['mc_paths.html', 'mc_distribution.html'],
 'naive_risk_parity': ['mc_paths.html', 'mc_distribution.html'],
 'hrp_style': ['mc_paths.html', 'mc_distribution.html']}

## 11. Batch export absolutely everything

In [17]:
batch_summary = viz.save_everything(max_mc_paths=50)
batch_summary

{'market_data_dir': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/data'),
 'constructions': {'equal_weight': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/constructions/equal_weight'),
  'markowitz': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/constructions/markowitz'),
  'naive_risk_parity': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/constructions/naive_risk_parity'),
  'hrp_style': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/constructions/hrp_style')},
 'backtests': {'equal_weight': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/backtests/equal_weight'),
  'markowitz': WindowsPath('C:/Users/narro/Desktop/semestre/honores_actuaria/outputs/runs/thesis_full_demo/backtests/markowitz'),
  'naive_risk_parity': Wi